Banking loan risk analysis

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Uploading datasets

In [25]:
customers = pd.read_csv("/content/bank_loan_risk_analysis/customers.csv")
loans = pd.read_csv("/content/bank_loan_risk_analysis/loans.csv")
payments = pd.read_csv("/content/bank_loan_risk_analysis/loan_payments.csv")

Merge Customer and Loan Data

In [26]:
df = loans.merge(
    customers,
    on="customer_id",
    how="left"
)

Data Quality Checking

In [27]:
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   loan_id          800 non-null    int64  
 1   customer_id      800 non-null    int64  
 2   branch_id        800 non-null    int64  
 3   loan_type        800 non-null    object 
 4   purpose          800 non-null    object 
 5   loan_amount      800 non-null    int64  
 6   tenure_months    800 non-null    int64  
 7   interest_rate    800 non-null    float64
 8   dti              800 non-null    float64
 9   loan_status      800 non-null    object 
 10  approval_status  800 non-null    object 
 11  customer_name    800 non-null    object 
 12  age              800 non-null    int64  
 13  annual_income    800 non-null    int64  
 14  city             800 non-null    object 
 15  occupation       800 non-null    object 
 16  credit_score     800 non-null    int64  
 17  existing_loans  

Create Credit Risk Bands

In [28]:
df["credit_band"] = pd.cut(
    df["credit_score"],
    bins=[0, 599, 679, 749, 900],
    labels=[
        "Poor",
        "Fair",
        "Good",
        "Excellent"
    ]
)

Create DTI Risk Bands

In [29]:
df["dti_band"] = pd.cut(
    df["dti"],
    bins=[-np.inf, 0.35, 0.50, np.inf],
    labels=[
        "Low",
        "Medium",
       "High"
    ]
)

Create Default Flag

In [30]:
df["default_flag"] = (
    df["loan_status"] == "Defaulted"
).astype(int)

Create High-Risk Flag

In [31]:
df["high_risk_flag"] = (
    (df["credit_score"] < 600) |
    (df["dti"] > 0.50) |
    (df["default_flag"] == 1)
).astype(int)


In [32]:
print(risk_by_credit)

  credit_band  loans  default_rate      avg_loan
0        Poor     87      0.287356  1.268448e+06
1        Fair    295      0.261017  1.312525e+06
2        Good    266      0.154135  1.344342e+06
3   Excellent    152      0.197368  1.343717e+06


Loan-Type Analysis

In [33]:
risk_by_type = df.groupby("loan_type").agg(
    loans=("loan_id", "count"),
    portfolio_value=("loan_amount", "sum"),
    default_rate=("default_flag", "mean")
).reset_index()

In [34]:
#Sort
risk_by_type = risk_by_type.sort_values(
    "portfolio_value",
    ascending=False
)

Payment Analysis

In [35]:
payment_summary = payments.groupby("loan_id").agg(
    payments=("payment_id", "count"),
    avg_days_late=("days_late", "mean"),
    late_payments=(
        "payment_status",
        lambda x: (x == "Late").sum()
    )
).reset_index()

payment_summary["late_rate"] = (
    payment_summary["late_payments"]
    / payment_summary["payments"]
)


In [36]:
#Merge It
df = df.merge(
    payment_summary,
    on="loan_id",
    how="left"
)

Export Power BI Dataset

In [37]:
df.to_csv(
    "powerbi_loan_risk.csv",
    index=False
)

payments.to_csv(
    "powerbi_payments.csv",
    index=False
)